# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-') and len(api_key) > 10:
    print('OpenAI API key found')
else:
    print('OpenAI API key missing or malformed; check your .env file')

openai_client = OpenAI()
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

OpenAI API key found


In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [6]:
# Get gpt-4o-mini to answer, with streaming

gpt_stream = openai_client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {
            'role': 'system',
            'content': 'You are a concise and helpful technical tutor. Explain clearly and include a short example when useful.'
        },
        {'role': 'user', 'content': question}
    ],
    stream=True
)

gpt_response = ''
display_handle = display(Markdown(''), display_id=True)

for chunk in gpt_stream:
    delta = chunk.choices[0].delta.content or ''
    gpt_response += delta
    update_display(Markdown(gpt_response), display_id=display_handle.display_id)

gpt_response

This line of code is using a generator to yield values derived from a set comprehension. Let's break it down:

1. **Set Comprehension**: The part `{book.get("author") for book in books if book.get("author")}` creates a set of unique authors from a collection called `books`. 
   - `book.get("author")`: This retrieves the value associated with the key "author" from each `book` dictionary.
   - `for book in books`: This iterates over each `book` in the `books` collection.
   - `if book.get("author")`: This is a filter that only includes the book's author if it exists (i.e., it's not `None` or an empty string).

2. **Yield from**: The `yield from` statement is used to yield all values from an iterable (in this case, the set created by the comprehension). This makes the function behave like a generator that can produce multiple values over time.

### Example 

Imagine you have the following list of book dictionaries:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": None},
    {"title": "Book 4", "author": "Author A"},
    {"title": "Book 5", "author": "Author C"},
]
```

The code `yield from {book.get("author") for book in books if book.get("author")}` will yield the unique authors:
- First, it will build the set: `{"Author A", "Author B", "Author C"}` (not including duplicates or `None`).
- Then, it will yield each author when the generator is iterated over.

### Summary
This line effectively creates a generator that produces unique author names from a list of books, while ignoring any books that do not have an author defined.

'This line of code is using a generator to yield values derived from a set comprehension. Let\'s break it down:\n\n1. **Set Comprehension**: The part `{book.get("author") for book in books if book.get("author")}` creates a set of unique authors from a collection called `books`. \n   - `book.get("author")`: This retrieves the value associated with the key "author" from each `book` dictionary.\n   - `for book in books`: This iterates over each `book` in the `books` collection.\n   - `if book.get("author")`: This is a filter that only includes the book\'s author if it exists (i.e., it\'s not `None` or an empty string).\n\n2. **Yield from**: The `yield from` statement is used to yield all values from an iterable (in this case, the set created by the comprehension). This makes the function behave like a generator that can produce multiple values over time.\n\n### Example \n\nImagine you have the following list of book dictionaries:\n\n```python\nbooks = [\n    {"title": "Book 1", "author": 

In [8]:
# Get Llama 3.2 to answer, with streaming

llama_stream = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {
            'role': 'system',
            'content': 'You are a concise and helpful technical tutor. Explain clearly and include a short example when useful.'
        },
        {'role': 'user', 'content': question}
    ],
    stream=True
)

llama_text = ''
display_handle = display(Markdown(''), display_id=True)

for chunk in llama_stream:
    delta = chunk.choices[0].delta.content or ''
    llama_text += delta
    update_display(Markdown(llama_text), display_id=display_handle.display_id)

llama_text

**Code Explanation**

This line of code is using Python's `yield from` keyword to delegate iteration over a generator expression.

```python
yield from {book.get("author") for book in books if book.get("author")}
```

Here's a breakdown:

* `{book.get("author") for book in books if book.get("author")}`: This is an iterable (a `set`, specifically) that yields the authors of books where the "author" key exists in the dictionary `book`. The `if` clause filters out books with missing or empty author information.
* `yield from`: This keyword tells Python to delegate the iteration over the set to another generator.

So, what happens?

When you iterate over this expression, it doesn't actually yield the values immediately. Instead, it yields a `GeneratorSet` (a special type of iterator that tracks its own state), which is then used by the `yield from` clause to retrieve and enumerate the authors.

In other words:

1. The `iteration` is done internally by Python for the generator expression.
2. Only when the iteration is requested again (`next()` is called, or the iterator is assigned to another variable) will `yield from` execute its internal state machine.

**Example**

Suppose you have a list of books with corresponding author data:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": None},
    {"title": "Book 3", "author": "Author C"}
]

for author in yield from {book.get("author") for book in books if book.get("author")}:
    print(author)
```

This code would output:

```
Author A
Author C
```

As expected, only the authors of complete entries are printed.

'**Code Explanation**\n\nThis line of code is using Python\'s `yield from` keyword to delegate iteration over a generator expression.\n\n```python\nyield from {book.get("author") for book in books if book.get("author")}\n```\n\nHere\'s a breakdown:\n\n* `{book.get("author") for book in books if book.get("author")}`: This is an iterable (a `set`, specifically) that yields the authors of books where the "author" key exists in the dictionary `book`. The `if` clause filters out books with missing or empty author information.\n* `yield from`: This keyword tells Python to delegate the iteration over the set to another generator.\n\nSo, what happens?\n\nWhen you iterate over this expression, it doesn\'t actually yield the values immediately. Instead, it yields a `GeneratorSet` (a special type of iterator that tracks its own state), which is then used by the `yield from` clause to retrieve and enumerate the authors.\n\nIn other words:\n\n1. The `iteration` is done internally by Python for the 